In [4]:
# handle the display
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [5]:
# imports
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("pyspark-sql-example").getOrCreate()


24/01/09 16:31:44 WARN Utils: Your hostname, nalain-dell resolves to a loopback address: 127.0.1.1; using 10.200.120.6 instead (on interface wlp0s20f3)
24/01/09 16:31:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/01/09 16:31:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
spark

In [12]:
path_to_file = Path("data/titanic.csv")

titanic = spark.read.csv(str(path_to_file),
                         header=True, 
                         inferSchema=True
                        )
titanic.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [13]:
titanic.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [14]:
## select 
titanic.select("Name","sex","Survived").show(5)

+--------------------+------+--------+
|                Name|   sex|Survived|
+--------------------+------+--------+
|Braund, Mr. Owen ...|  male|       0|
|Cumings, Mrs. Joh...|female|       1|
|Heikkinen, Miss. ...|female|       1|
|Futrelle, Mrs. Ja...|female|       1|
|Allen, Mr. Willia...|  male|       0|
+--------------------+------+--------+
only showing top 5 rows



In [15]:
## group by

titanic.select("Name","sex","Survived").groupBy("sex").sum().show(5)

+------+-------------+
|   sex|sum(Survived)|
+------+-------------+
|female|          233|
|  male|          109|
+------+-------------+



# sql and spark

Once the view is created we can utilize sql command view spark sql context

In [18]:
# create a temporarily view

titanic.createOrReplaceTempView("titanic")

In [19]:
## now we can use any sql statements

spark.sql("SELECT * FROM titanic;").show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [25]:
# survived count per gender

spark.sql("""
            SELECT 
                Sex, 
                ROUND(AVG(age),2) as MEAN_AGE
            FROM titanic
            GROUP BY sex
            """
         ).show(5)

+------+--------+
|   Sex|MEAN_AGE|
+------+--------+
|female|   27.92|
|  male|   30.73|
+------+--------+



In [ ]:
# aggregate age
spark.sql("""
            SELECT 
                Sex, 
                Sum(survived)
            FROM titanic
            GROUP BY sex
            """
         ).show(5)

In [27]:
# populate
spark.sql("""
            SELECT 
                *, 
                CASE 
                    WHEN Cabin IS NOT NULL THEN 1
                    ELSE 0
                END AS HasCabin
            FROM titanic
            """
         ).show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|HasCabin|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|       0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|       1|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|       0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|       1|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|       0|
+-------

In [30]:
# sql and regular exp

# populate
spark.sql("""
            SELECT 
                Name,
                REGEXP_EXTRACT(name, ' ([A-Za-z]+)\.') AS Title
            FROM titanic
            """
         ).show(5)

+--------------------+-----+
|                Name|Title|
+--------------------+-----+
|Braund, Mr. Owen ...|   Mr|
|Cumings, Mrs. Joh...|  Mrs|
|Heikkinen, Miss. ...| Miss|
|Futrelle, Mrs. Ja...|  Mrs|
|Allen, Mr. Willia...|   Mr|
+--------------------+-----+
only showing top 5 rows



In [31]:
# USER DEFINED FUNCTIONS

def reverse_name(name):
    return name[::-1]
reverse_name("alain patrick ndigande")

'ednagidn kcirtap niala'

In [32]:
# register the functions

spark.udf.register("reverse_name",reverse_name)

<function __main__.reverse_name(name)>

In [33]:
spark.sql("""
SELECT 
    name,
    reverse_name(name) as ReversedName
FROM titanic
""").show(5)

+--------------------+--------------------+
|                name|        ReversedName|
+--------------------+--------------------+
|Braund, Mr. Owen ...|sirraH newO .rM ,...|
|Cumings, Mrs. Joh...|)reyahT sggirB ec...|
|Heikkinen, Miss. ...|aniaL .ssiM ,neni...|
|Futrelle, Mrs. Ja...|)leeP yaM yliL( h...|
|Allen, Mr. Willia...|yrneH mailliW .rM...|
+--------------------+--------------------+
only showing top 5 rows



In [34]:
spark.stop()